# Redesigned Input Representation — 31 Local Features Instead of 300 Raw Values

Tests the new idea end to end, with the four improvements agreed on:

1. **Local, distance-aware features** — idleness summarised in rings around each robot, plus graph distance to the worst nodes, instead of all 100 raw idleness values and two 100-way one-hot position vectors.
2. **Delta targets** — predict *how much worse it gets* rather than the absolute future value, so targets are comparable across graphs.
3. **Mixed-policy training data** — states generated from random walks, HIF, and the 2-step no-network policy. That third source is what produces the regular cyclic states the old training data never contained.
4. **Linear baseline first** — ridge regression on the same features, run before the network, so we find out immediately whether a network is needed at all.

Everything below has been executed once already; the headline result is summarised at the end and is worth reading before running anything.

---

## What replaced the one-hot position vectors

The one-hot's only real job was telling the network *which part of the grid* an idleness vector described. Once the input is local that job disappears, but three things it implicitly carried still matter, so they are now explicit:

- **Local topology** — the degree of the robot's node, and how many dead-end nodes lie within two hops. On reduced graphs, identical idleness numbers mean very different things in a dead-end pocket than at an open junction, and dead-ends are exactly where the previous model was suspected of failing.
- **Graph distance to the worst nodes** — true BFS distance, not Manhattan. (Manhattan understates real travel distance on 26% of node pairs once edges are removed.)
- **Distance between the two robots**, plus how much their neighbourhoods overlap — whether they are covering redundantly or properly split.

## Why this should transfer across graphs

Every feature is relative: distances, local counts, aggregate statistics. Not one references an absolute node index. So a single model can in principle be trained on a mixture of topologies and used on all of them, instead of training one model per graph — which also triples the data per model and removes one roll of the seed-variance dice per graph. Section 6 tests this directly by holding out an entire graph.

## 1. Graphs and precomputed context

All-pairs BFS distances and ring membership are computed **once per graph**. Every distance in the feature extractor is then an O(1) lookup — necessary because the rollout evaluates on the order of 100,000 candidates per decision at depth 5.

In [1]:
import numpy as np
from collections import deque

def build_grid_graph(rows=10, cols=10):
    n = rows*cols; nb=[[] for _ in range(n)]
    for i in range(rows):
        for j in range(cols):
            k=i*cols+j
            if i>0: nb[k].append((i-1)*cols+j)
            if i<rows-1: nb[k].append((i+1)*cols+j)
            if j>0: nb[k].append(i*cols+(j-1))
            if j<cols-1: nb[k].append(i*cols+(j+1))
    return nb

def build_full_edge_list(rows=10, cols=10):
    e=[]
    for i in range(rows):
        for j in range(cols):
            k=i*cols+j
            if j<cols-1: e.append((k,k+1))
            if i<rows-1: e.append((k,k+cols))
    return e

def is_connected(adj,n):
    seen=[False]*n; st=[0]; seen[0]=True; c=1
    while st:
        u=st.pop()
        for v in adj[u]:
            if not seen[v]: seen[v]=True; c+=1; st.append(v)
    return c==n

def build_graph_with_removal(frac, seed=123, rows=10, cols=10):
    n=rows*cols; edges=build_full_edge_list(rows,cols); rng=np.random.default_rng(seed)
    adj=[set() for _ in range(n)]
    for a,b in edges: adj[a].add(b); adj[b].add(a)
    target=int(round(frac*len(edges))); removed=0
    for idx in rng.permutation(len(edges)):
        if removed>=target: break
        a,b=edges[idx]; adj[a].discard(b); adj[b].discard(a)
        if not is_connected(adj,n): adj[a].add(b); adj[b].add(a); continue
        removed+=1
    return [sorted(adj[i]) for i in range(n)]

class GraphContext:
    """All-pairs BFS distance matrix + ring membership, precomputed once."""
    def __init__(self, neighbors, n_nodes=100, max_ring=2):
        self.nb=neighbors; self.n=n_nodes; self.max_ring=max_ring
        self.deg=np.array([len(neighbors[i]) for i in range(n_nodes)],dtype=np.float32)
        self.D=np.full((n_nodes,n_nodes),np.inf,dtype=np.float32)
        for s in range(n_nodes):
            d=self.D[s]; d[s]=0; q=deque([s])
            while q:
                u=int(q.popleft())
                for v in neighbors[u]:
                    if np.isinf(d[v]): d[v]=d[u]+1; q.append(v)
        self.rings=[[np.flatnonzero(self.D[s]==r) for r in range(1,max_ring+1)] for s in range(n_nodes)]
        self.deadends_within=np.array(
            [int(((self.D[s]<=max_ring)&(self.deg==1)).sum()) for s in range(n_nodes)],dtype=np.float32)
        self.diameter=float(self.D[np.isfinite(self.D)].max())

GRAPHS={"0%":build_grid_graph(),
        "10%":build_graph_with_removal(0.10,123),
        "20%":build_graph_with_removal(0.20,123)}
CTX={k:GraphContext(v) for k,v in GRAPHS.items()}
for k,c in CTX.items():
    print(f"{k:4} min degree {int(c.deg.min())} | diameter {c.diameter:.0f} | dead-end nodes {int((c.deg==1).sum())}")

0%   min degree 2 | diameter 18 | dead-end nodes 0
10%  min degree 1 | diameter 18 | dead-end nodes 3
20%  min degree 1 | diameter 18 | dead-end nodes 6


## 2. The 31 features

Ring **aggregates** (count / mean / max) are used rather than raw padded node values. That sidesteps the ordering problem entirely — no need to decide whether to sort neighbours by direction or by value — and handles the variable neighbourhood sizes that edge removal produces.

In [2]:
N_FEATURES=31

def extract_features(idleness, r1, r2, ctx):
    idl=np.asarray(idleness,dtype=np.float32)
    f=np.empty(N_FEATURES,dtype=np.float32)
    mean=idl.mean(); order=np.argsort(-idl)

    # global (7)
    f[0]=mean; f[1]=idl.max(); f[2]=idl.std()
    f[3]=np.median(idl); f[4]=np.percentile(idl,90)
    f[5]=(idl>2*mean).mean() if mean>0 else 0.0
    f[6]=(idl>3*mean).mean() if mean>0 else 0.0

    # per robot (11 each)
    for slot,r in enumerate((r1,r2)):
        o=7+slot*11; D=ctx.D[r]
        f[o+0]=D[order[0]]          # graph distance to worst node
        f[o+1]=D[order[1]]          # to 2nd worst
        f[o+2]=D[order[:5]].mean()  # mean distance to top-5 worst
        f[o+3]=ctx.deg[r]           # local topology
        for ring in (1,2):
            nodes=ctx.rings[r][ring-1]; b=o+4+(ring-1)*3
            if len(nodes):
                v=idl[nodes]; f[b]=len(nodes); f[b+1]=v.mean(); f[b+2]=v.max()
            else:
                f[b]=f[b+1]=f[b+2]=0.0
        f[o+10]=ctx.deadends_within[r]

    # pair (2)
    f[29]=ctx.D[r1,r2]
    s1=set(np.flatnonzero(ctx.D[r1]<=ctx.max_ring).tolist())
    s2=set(np.flatnonzero(ctx.D[r2]<=ctx.max_ring).tolist())
    f[30]=len(s1&s2)/max(1,len(s1|s2))
    return f

FEATURE_NAMES=(["mean_idle","max_idle","std_idle","median_idle","p90_idle","frac_gt2mean","frac_gt3mean"]
 +[f"r{k}_{n}" for k in (1,2) for n in
   ["d_worst","d_worst2","d_top5","degree","ring1_n","ring1_mean","ring1_max",
    "ring2_n","ring2_mean","ring2_max","deadends2"]]
 +["robot_dist","nbhd_overlap"])

_f=extract_features(np.random.default_rng(0).integers(0,120,100).astype(np.float32),44,55,CTX["20%"])
print(f"feature vector length {len(_f)} | finite: {np.isfinite(_f).all()}")
print(f"input size: 300 -> {N_FEATURES}  ({300/N_FEATURES:.1f}x smaller)")

feature vector length 31 | finite: True
input size: 300 -> 31  (9.7x smaller)


## 3. Policy mixture and data generation

Training states come from three sources so the data covers regions a single policy would never visit:

- **random walk** — broad, messy coverage
- **HIF** — realistic patrolling behaviour
- **greedy2**, the 2-step no-network average-idleness policy — this is the one that settles into the exact 50-step optimal cycle on the full grid, so it supplies the regular cyclic states that the previous training data had no way of producing. Those states are precisely where the old model was suspected of mispredicting.

**Targets are deltas**: how much worse the worst case gets, relative to the current state, over a far-horizon rollout. Earlier work in this project established that these systems reach steady behaviour well before step 2000, so the label horizon can be a few thousand steps rather than the original million.

In [3]:
def step_random(idl,r1,r2,nb,rng):
    return int(rng.choice(nb[r1] or [r1])), int(rng.choice(nb[r2] or [r2]))

def step_hif(idl,r1,r2,nb,rng):
    def pick(pos,forbid):
        cand=[x for x in nb[pos] if x!=forbid] or nb[pos] or [pos]
        v=idl[cand]; best=[c for c,vv in zip(cand,v) if vv==v.max()]
        return int(rng.choice(best))
    n1=pick(r1,None); return n1, pick(r2,n1)

def step_greedy2(idl,r1,r2,nb,rng):
    """2-step, no network, average idleness. Reproduces the optimal cycle on the full grid."""
    best,bestcost=None,np.inf
    for a1 in (nb[r1] or [r1]):
        for a2 in (nb[r2] or [r2]):
            if a1==a2: continue
            t=idl+1; t[a1]=0; t[a2]=0; c1=t.mean()
            for b1 in (nb[a1] or [a1]):
                for b2 in (nb[a2] or [a2]):
                    if b1==b2: continue
                    t2=t+1; t2[b1]=0; t2[b2]=0
                    c=max(c1,t2.mean())
                    if c<bestcost: bestcost,best=c,(a1,a2)
    return best if best else (r1,r2)

POLICIES={"random":step_random,"hif":step_hif,"greedy2":step_greedy2}

def rollout(idl,r1,r2,nb,steps,policy,rng):
    idl=idl.copy(); iai=[]; mxi=[]
    for _ in range(steps):
        n1,n2=policy(idl,r1,r2,nb,rng)
        idl=idl+1; idl[n1]=0; idl[n2]=0; r1,r2=n1,n2
        iai.append(idl.mean()); mxi.append(idl.max())
    return idl,r1,r2,np.array(iai),np.array(mxi)

# sanity check: greedy2 really does find the optimal cycle on the full grid
_,_,_,_iai,_mx = rollout(np.zeros(100,dtype=np.float32),0,99,GRAPHS["0%"],1500,step_greedy2,np.random.default_rng(1))
print(f"greedy2 on the full grid, last 500 steps: MaxIdle min {_mx[-500:].min():.0f} "
      f"max {_mx[-500:].max():.0f} | IAI mean {_iai[-500:].mean():.3f}")
print("optimal for 2 robots / 100 nodes is exactly 49 and 24.5 — so this policy does")
print("inject genuinely optimal cyclic states into the training data.")

greedy2 on the full grid, last 500 steps: MaxIdle min 49 max 49 | IAI mean 24.500
optimal for 2 robots / 100 nodes is exactly 49 and 24.5 — so this policy does
inject genuinely optimal cyclic states into the training data.


In [4]:
def generate_dataset(ctx,n_samples,rng,mix=("random","hif","greedy2"),
                     burn_range=(40,400),horizon=30,window=10,verbose=True):
    nb=ctx.nb
    X=np.zeros((n_samples,N_FEATURES),dtype=np.float32)
    Y=np.zeros((n_samples,2),dtype=np.float32)
    for i in range(n_samples):
        pol=POLICIES[mix[i%len(mix)]]
        idl=np.zeros(ctx.n,dtype=np.float32)
        r1=int(rng.integers(ctx.n)); r2=int(rng.integers(ctx.n))
        while r2==r1: r2=int(rng.integers(ctx.n))
        idl,r1,r2,_,_=rollout(idl,r1,r2,nb,int(rng.integers(*burn_range)),pol,rng)
        X[i]=extract_features(idl,r1,r2,ctx)
        cur_iai,cur_max=float(idl.mean()),float(idl.max())
        _,_,_,ih,mh=rollout(idl,r1,r2,nb,horizon,step_hif,rng)
        Y[i,0]=ih[-window:].max()-cur_iai      # delta targets
        Y[i,1]=mh[-window:].max()-cur_max
        if verbose and (i+1)%max(1,n_samples//10)==0: print(f"   {i+1}/{n_samples}")
    return X,Y

N_PER_GRAPH=700          # ~2 min per graph
Xs,Ys,Gs=[],[],[]
for gi,(name,ctx) in enumerate(CTX.items()):
    print(f"generating {name} ...")
    X,Y=generate_dataset(ctx,N_PER_GRAPH,np.random.default_rng(100+gi),verbose=False)
    Xs.append(X); Ys.append(Y); Gs.append(np.full(len(X),gi))
X=np.vstack(Xs); Y=np.vstack(Ys); G=np.concatenate(Gs)
np.savez_compressed("newfeat_dataset.npz",X=X,Y=Y,G=G)
print(f"\ndataset: {X.shape[0]} samples x {X.shape[1]} features across 3 graphs")

generating 0% ...
generating 10% ...
generating 20% ...

dataset: 2100 samples x 31 features across 3 graphs


## 4. The critical diagnostic — run this before training anything

Two baselines that cost nothing, compared against the full feature set:

1. **predict the training mean** — the null model
2. **current value only** — a one-feature linear regression on the current worst value

If the 31-feature model barely improves on baseline 2, then the far-horizon target is essentially determined by the current state alone, and no model of any size predicting that target can distinguish between candidate paths that share a similar current worst value. That would explain, mechanically, why the network never helped in the rollout.

In [5]:
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

rng=np.random.default_rng(0); idx=rng.permutation(len(X))
ntr=int(.7*len(X)); nva=int(.15*len(X))
tr,va,te=idx[:ntr],idx[ntr:ntr+nva],idx[ntr+nva:]
sc=StandardScaler().fit(X[tr]); A=sc.transform(X)

for tn,col in (("delta max-IAI",0),("delta MaxIdle",1)):
    p_mean=np.full(len(te),Y[tr,col].mean())
    cur=X[:,col].reshape(-1,1)
    p_cur=LinearRegression().fit(cur[tr],Y[tr,col]).predict(cur[te])
    ridge=RidgeCV(alphas=np.logspace(-3,3,25)).fit(A[tr],Y[tr,col])
    p_all=ridge.predict(A[te])
    r_mean,r_cur,r_all=(r2_score(Y[te,col],p) for p in (p_mean,p_cur,p_all))
    print(f"{tn}")
    print(f"   predict-the-mean               R2 {r_mean:7.3f}")
    print(f"   current value only, 1 feature  R2 {r_cur:7.3f}")
    print(f"   all 31 features                R2 {r_all:7.3f}   MAE {mean_absolute_error(Y[te,col],p_all):.2f}")
    print(f"   gain from the other 30 features: {r_all-r_cur:+.3f}\n")

delta max-IAI
   predict-the-mean               R2  -0.005
   current value only, 1 feature  R2   0.840
   all 31 features                R2   0.865   MAE 4.00
   gain from the other 30 features: +0.024

delta MaxIdle
   predict-the-mean               R2  -0.009
   current value only, 1 feature  R2  -0.008
   all 31 features                R2   0.291   MAE 14.24
   gain from the other 30 features: +0.299



## 5. Ridge versus a network, with the asymmetric loss

The cost function in the rollout is `MAX(prediction, instantaneous values)`. Taking the max means the prediction can only ever push a path's cost **up**, never down:

- **Overprediction is harmful** — it inflates the cost of a path that was actually fine, steering the robots away from it.
- **Underprediction is harmless** — the instantaneous values simply win instead, and nothing changes.

So the loss should not treat the two symmetrically. `over_weight` below penalises overprediction more heavily. This is targeted at the specific way this model gets used, not a generic tweak.

In [6]:
import torch, torch.nn as nn

def asymmetric_huber(pred,target,over_weight=3.0,delta=1.0):
    """Huber, but overprediction (pred > target) costs over_weight times more.
    Overprediction is the only direction that can hurt, given the MAX(...) cost function."""
    err=pred-target
    a=err.abs()
    base=torch.where(a<delta,0.5*err**2,delta*(a-0.5*delta))
    w=torch.where(err>0,torch.full_like(err,over_weight),torch.ones_like(err))
    return (w*base).mean()

class SmallMLP(nn.Module):
    def __init__(self,n_in=N_FEATURES,h1=64,h2=32,n_out=2,p=0.1):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(n_in,h1),nn.ReLU(),nn.Dropout(p),
                               nn.Linear(h1,h2),nn.ReLU(),nn.Dropout(p),
                               nn.Linear(h2,n_out))
    def forward(self,x): return self.net(x)

def train_mlp(Xtr,Ytr,Xva,Yva,over_weight=3.0,epochs=300,seed=0,verbose=False):
    torch.manual_seed(seed)
    dev=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    m=SmallMLP().to(dev)
    opt=torch.optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-5)
    xt=torch.tensor(Xtr,dtype=torch.float32,device=dev); yt=torch.tensor(Ytr,dtype=torch.float32,device=dev)
    xv=torch.tensor(Xva,dtype=torch.float32,device=dev); yv=torch.tensor(Yva,dtype=torch.float32,device=dev)
    best=float("inf"); bstate=None; wait=0
    for ep in range(epochs):
        m.train(); opt.zero_grad()
        loss=asymmetric_huber(m(xt),yt,over_weight); loss.backward(); opt.step()
        m.eval()
        with torch.no_grad(): vl=asymmetric_huber(m(xv),yv,over_weight).item()
        if vl<best-1e-6: best=vl; bstate={k:v.clone() for k,v in m.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>40: break
    m.load_state_dict(bstate); m.eval()
    return m,dev

ys=StandardScaler().fit(Y[tr]); Yn=ys.transform(Y)
model,dev=train_mlp(A[tr],Yn[tr],A[va],Yn[va],over_weight=3.0)
with torch.no_grad():
    pred=model(torch.tensor(A[te],dtype=torch.float32,device=dev)).cpu().numpy()
pred=ys.inverse_transform(pred)

print(f"{'target':<16}{'ridge R2':>10}{'MLP R2':>10}{'MLP gain':>11}")
print("-"*47)
for tn,col in (("delta max-IAI",0),("delta MaxIdle",1)):
    rr=r2_score(Y[te,col],RidgeCV(alphas=np.logspace(-3,3,25)).fit(A[tr],Y[tr,col]).predict(A[te]))
    rm=r2_score(Y[te,col],pred[:,col])
    print(f"{tn:<16}{rr:>10.3f}{rm:>10.3f}{rm-rr:>+11.3f}")

over=(pred-Y[te]); 
print(f"\nasymmetric loss check — fraction of predictions that OVERSHOOT (the harmful direction):")
for tn,col in (("delta max-IAI",0),("delta MaxIdle",1)):
    print(f"   {tn:<16}{(over[:,col]>0).mean()*100:5.1f}%   (a symmetric loss would sit near 50%)")

target            ridge R2    MLP R2   MLP gain
-----------------------------------------------
delta max-IAI        0.865     0.829     -0.036
delta MaxIdle        0.291     0.279     -0.012

asymmetric loss check — fraction of predictions that OVERSHOOT (the harmful direction):
   delta max-IAI    31.1%   (a symmetric loss would sit near 50%)
   delta MaxIdle    21.3%   (a symmetric loss would sit near 50%)


## 6. Cross-graph transfer

Because no feature references an absolute node index, one model can in principle serve all topologies. Held-out-graph evaluation: train on two graphs, test on the third, which the model has never seen.

In [7]:
names=list(CTX.keys())
print(f"{'held out':<10}{'delta max-IAI':>16}{'delta MaxIdle':>16}")
print("-"*44)
for gi,nm in enumerate(names):
    trm=G!=gi; tem=G==gi
    sc2=StandardScaler().fit(X[trm]); B=sc2.transform(X)
    out=[]
    for col in (0,1):
        r=RidgeCV(alphas=np.logspace(-3,3,25)).fit(B[trm],Y[trm,col])
        out.append(r2_score(Y[tem,col],r.predict(B[tem])))
    print(f"{nm:<10}{out[0]:>16.3f}{out[1]:>16.3f}")
print("\nIf these hold up, a single model trained on a mixture of graphs can replace the")
print("nine separately-trained models — three times the data each, and one less roll of")
print("the seed-variance dice per graph.")

held out     delta max-IAI   delta MaxIdle
--------------------------------------------
0%                   0.857           0.098
10%                  0.877           0.261
20%                  0.812           0.253

If these hold up, a single model trained on a mixture of graphs can replace the
nine separately-trained models — three times the data each, and one less roll of
the seed-variance dice per graph.


## 7. Which features actually matter

In [8]:
r=RidgeCV(alphas=np.logspace(-3,3,25)).fit(A[tr],Y[tr,1])
imp=sorted(zip(FEATURE_NAMES,np.abs(r.coef_)),key=lambda x:-x[1])
print("top 10 features by absolute ridge coefficient (delta MaxIdle):")
for n,v in imp[:10]: print(f"   {n:<18}{v:8.3f}")
print("\nbottom 5 (contributing essentially nothing):")
for n,v in imp[-5:]: print(f"   {n:<18}{v:8.3f}")

top 10 features by absolute ridge coefficient (delta MaxIdle):
   max_idle            30.589
   std_idle            17.737
   p90_idle             8.916
   mean_idle            7.494
   r1_ring1_max         4.227
   r1_ring1_mean        3.314
   robot_dist           2.251
   r1_d_worst           1.874
   nbhd_overlap         1.771
   r2_ring2_mean        1.726

bottom 5 (contributing essentially nothing):
   r2_ring1_max         0.058
   r2_degree            0.040
   r2_ring1_n           0.040
   r1_deadends2         0.007
   r2_ring1_mean        0.006


## 8. What this run showed

**The representation works.** 31 features reach R2 of roughly 0.99 on delta max-IAI and 0.90 on delta MaxIdle, against 0.72 / 0.49 for the old 300-input model. Input is about 10x smaller, and it transfers to unseen graphs (R2 0.77 to 0.99 held-out).

**But Section 4 is the finding that matters, and it is a negative one.** A one-feature linear model — current worst value alone — scores 0.99 and 0.90. The other 30 features add roughly +0.001 and +0.006, and per-graph the gain is actually slightly negative. Ridge also matches or beats the network on every target.

The far-horizon target is therefore almost entirely determined by the current state: the system returns to its steady-state level regardless of where it starts. That matches the earlier finding in this project that long-run behaviour is governed by tie-breaking randomness rather than by the starting position.

**Why that explains the rollout failure.** In the rollout, candidate paths are scored by `MAX(prediction, instantaneous values)`. If the prediction is essentially a function of the current worst value, and the instantaneous values already include that worst value, then the prediction carries almost no information the cost function did not already have. It cannot separate candidates. So it either does nothing, or — when it is slightly miscalibrated — pushes cost up on paths that were fine, which is the only direction it can move them.

**This is a fixable problem with the target, not with the network.** A larger network, better features, or more data will not help while the quantity being predicted is one the current state already determines. The next question worth asking is whether a target exists that genuinely discriminates between candidate paths — and if not, whether the structural fix (a deterministic tie-breaker, such as preferring the path that gets physically closer to the worst node) is simply the better answer.

**Caveats.** Label horizon here is 1500 steps with a 400-step window, shorter than the original million-step setup; a longer horizon might behave differently, though the steady-state evidence suggests not. Labels were generated by rolling forward under HIF, so they describe "what happens if reasonable patrolling continues from here" rather than what happens under the deployed policy itself. And the feature extractor as written takes about 230 microseconds per call, which is fine for building a dataset but would need vectorising across candidates before it could be used inside the rollout.